In [1]:
import os
import torch

from tqdm import tqdm

from stock_gpt import StockGPT, LinearModel, NaiveModel
from dataloader_builder import build_dataloaders
from setup import StockGPT_cfg, LinearModel_cfg, NaiveModel_cfg
from setup import path_data_preprocessor

In [2]:
cuda = True if torch.cuda.is_available() else False

print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PyTorch: 2.13.0+cu132
CUDA build: 13.2
CUDA available: True
GPU: NVIDIA GeForce RTX 4070 Laptop GPU


In [3]:
def batch_loss(x, y, model, fn, weights = None):
    """
    Assume x and y are on the correct device already
    Applies weights and fn to the returned loss
    """
    x = model(x)
    y_norm = (y - model.target_mean) / model.target_std
    loss = fn(x, y_norm)
    if weights is not None:
        loss *= weights
    return loss

def loader_loss(data_loader, model, device, fns: dict = {}, weights = None, 
                max_batches = float("inf"), pbar = None, desc=""):
    """
    Returns a dict of loss calcualted using all loss functions in fns
    """
    num_batches = min(len(data_loader), max_batches)
    avg_metrics = {f: 0 for f in fns}
    for i, (p, t) in enumerate(data_loader):
        p = p.to(device, non_blocking=True)
        t = t.to(device, non_blocking=True)
        if i == num_batches:
            break
        for name, fn in fns.items():
            avg_metrics[name] += (batch_loss(p, t, model, fn, weights) - avg_metrics[name])/(i+1)
        if pbar is not None:
            pbar.update(1)
            if i % max(1,int(num_batches*0.001))==0:
                pbar.set_description(f"{desc} ({i}/{num_batches}) [{pbar.n}/{pbar.total}]")
    return avg_metrics

## MODEL TRAINING ---------------------------

In [ ]:
def load_model(path, model, device, optimizer=None, cuda_scaler=None):
    checkpoint = torch.load(path, map_location=device)
    model.load_state_dict(checkpoint["model"])
    if optimizer is not None and "optimizer" in checkpoint:
        optimizer.load_state_dict(checkpoint["optimizer"])
    if cuda_scaler is not None and "cuda_scaler" in checkpoint:
        cuda_scaler.load_state_dict(checkpoint["cuda_scaler"])
    return checkpoint

def evaluate_model(train_dl, val_dl, model, device, eval_fns, weights, eval_bs, pbar = None):
    """
    Returns a list of dictionaries, with each dictionary correspoding to a function in eval_fns
    """
    with torch.no_grad():
        train_metrics = loader_loss(train_dl, model, device, eval_fns, weights, eval_bs, pbar,
                                    desc="Evaluating model on training data...")
        val_metrics = loader_loss(val_dl, model, device, eval_fns, weights, eval_bs, pbar,
                                  desc="Evaluating model on validation data...")    
    return train_metrics, val_metrics

def evaluate_best_model(model, device, optimizer, cuda_scaler, train_dl, val_dl,
                        eval_fns, weights, eval_bs, pbar = None, evaluate = False):
    if os.path.exists(model.best_path):
        checkpoint = load_model(model.best_path, model, device, optimizer, cuda_scaler)
        if evaluate is False:
            return checkpoint["train_losses"][-1], checkpoint["val_losses"][-1]
        else:
            evaluate_model(train_dl, val_dl, model, device, eval_fns, weights, eval_bs, pbar)
    raise FileNotFoundError("Best parameters of the model could not be found")

def train_model_cuda(model, device, optimizer, cuda_scaler, max_epochs,
                     train_dl, val_dl, train_fn, eval_fns, weights, eval_bs):
    #* LOADS MODEL
    if os.path.exists(model.checkpoint_path):
        checkpoint = load_model(model.checkpoint_path, model, device, optimizer, cuda_scaler)
        bvm, epoch, train_losses, val_losses = (
            checkpoint["bvm"], checkpoint["epoch"]+1, checkpoint["train_losses"], checkpoint["val_losses"]
        )
    else:
        bvm, epoch, train_losses, val_losses = float("inf"), 0, [], []
    print("Linear checkpoint:", linearModel.checkpoint_path)
    print("Exists:", os.path.exists(linearModel.checkpoint_path))
    print("Train batches:", len(dls["train"]))
    print("Max epochs:", max_epochs)
    eval_steps = min(eval_bs, len(train_dl)) + min(eval_bs, len(val_dl))
    pbar = tqdm(total=(max_epochs-epoch)*(len(train_dl)+eval_steps), desc=f"Setting up...".ljust(80),
                bar_format="|{bar}| {percentage:3.1f}% ({elapsed}) {desc}", position=0, leave=False)
    pbar.write((f"Epoch {epoch+1}:\n"))
    try:
        for epoch in range(epoch, max_epochs):
            #* TRAINS MODEL
            model.train()
            for x, y in train_dl:
                x = x.to(device, non_blocking=True)
                y = y.to(device, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)

                with torch.autocast(device_type="cuda",dtype=torch.float16):
                    loss = batch_loss(x, y, model, train_fn).mean()
                cuda_scaler.scale(loss).backward()
                cuda_scaler.step(optimizer)
                cuda_scaler.update()

                pbar.update(1)
                if (pbar.n % max(1,int(pbar.total*0.001))==0):
                    pbar.set_description(f"Training the {model.cfg['name']}... [{pbar.n}/{pbar.total}]")

            #* EVALUATES MODEL
            model.eval()
            pbar.set_description(f"Evaluating Epoch {epoch}... [{pbar.n}/{pbar.total}]")
            train_metrics, val_metrics = evaluate_model(train_dl, val_dl, model, device,
                                                        eval_fns, weights, eval_bs, pbar)
            pbar.write((f"Epoch {epoch+1}:\n"
                        f"Training Loss = {train_metrics['MAE Loss'].mean()}\n"
                        f"Validation Loss = {val_metrics['MAE Loss'].mean()}\n"
                        ))
            train_losses.append(train_metrics)
            val_losses.append(val_metrics)

            #* SAVES MODEL
            cvm = val_metrics['MAE Loss'].mean().item()
            checkpoint = {
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "cuda_scaler": cuda_scaler.state_dict(),
                "epoch": epoch,
                "train_losses": train_losses,
                "val_losses": val_losses,
                "bvm": bvm
            }
            if (cvm < bvm):
                bvm = cvm
                checkpoint["bvm"] = cvm
                torch.save(checkpoint, model.best_path)
            pbar.write((f"Best Validation: {bvm} \n{'-'*100}\n"))
            torch.save(checkpoint, model.checkpoint_path)
    finally:
            pbar.close()
    print("Finished")
    return train_losses, val_losses

In [5]:
torch.manual_seed(1234)
dls, train_norms = build_dataloaders(path_data_preprocessor)

Building DataLoaders...


In [8]:
def model_setup(model_cls, cfg, train_norms, device, optimizer_cls, lr, weight_decay, scaler_cls, scale_type):
    model = model_cls(cfg, train_norms)
    model.to(device)
    model_params = sum(p.numel() for p in model.parameters())
    print(model_params)
    optimizer = optimizer_cls(model.parameters(), lr=lr, weight_decay=weight_decay)
    scaler = scaler_cls(scale_type)
    return model, model_params, optimizer, scaler

optimizer_data = [torch.optim.AdamW, 0.0004, 0.1]
scaler_data = [torch.amp.GradScaler, "cuda"]

train_fn = torch.nn.HuberLoss(reduction="none")
eval_fns = {
    "Huber Loss": torch.nn.HuberLoss(reduction="none"),
    "MAE Loss": torch.nn.L1Loss(reduction="none")
}

target_weights = torch.tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
    dtype=torch.float32,
    device=device
)
target_weights /= target_weights.mean()

max_epochs = 10
eval_bs = 500

stockGPT, stockGPT_params, o1, s1 = model_setup(StockGPT, StockGPT_cfg, train_norms, device,
                                                *optimizer_data, *scaler_data)
linearModel, linearModel_params, o2, s2 = model_setup(LinearModel, LinearModel_cfg, train_norms, device, 
                                                      *optimizer_data, *scaler_data)
naiveModel = NaiveModel(NaiveModel_cfg, train_norms)
naiveModel.to(device)


linear_train_losses, linear_val_losses = train_model_cuda(linearModel, device, o2, s2, max_epochs,
                                                        dls["train"], dls["val"], train_fn, eval_fns, target_weights, eval_bs)
model_train_losses, model_val_losses = train_model_cuda(stockGPT, device, o1, s1, max_epochs, 
                                                        dls["train"], dls["val"], train_fn, eval_fns, target_weights, eval_bs)

Input Norm: torch.Size([13])|torch.Size([13])
Target Norm: torch.Size([10])|torch.Size([10])
3182336
6144
Linear checkpoint: model_parameters/checkpoint_linear_model_5min
Exists: True
Train batches: 3691
Max epochs: 10


Epoch 11:

Finished
Linear checkpoint: model_parameters/checkpoint_linear_model_5min
Exists: True
Train batches: 3691
Max epochs: 10


|          | 0.0% (00:00) Setting up...                                                                   

Epoch 1:



|█         | 10.0% (05:15) Evaluating model on validation data... (499/500) [4691/46910]:                 

...

|████████  | 80.0% (42:16) Evaluating model on validation data... (499/500) [37528/46910]: 

Epoch 8:
Training Loss = 0.04007066786289215
Validation Loss = 0.03885914757847786
----------------------------------------------------------------------------------------------------

Best Validation MEA: 0.03885914757847786


|█████████ | 90.0% (47:31) Evaluating model on validation data... (499/500) [42219/46910]: 

Epoch 9:
Training Loss = 0.03935912996530533
Validation Loss = 0.03809250518679619
----------------------------------------------------------------------------------------------------

Best Validation MEA: 0.03809250518679619


Epoch 10:
Training Loss = 0.040218841284513474
Validation Loss = 0.038999173790216446
----------------------------------------------------------------------------------------------------

Finished


## Model Analysis -------------------------

In [9]:
def process_losses(losses: list[dict], key = "MAE Loss"):
    return [loss_dict[key].mean(dim=(0,1)) for loss_dict in losses]

def tensor_to_string(t, cs):
    return "".join(f"{v.item():<{cs}.4f}" for v in t)

def format_num(n):
    if n >= 1e9:
        return f"{n / 1e9:.1f}B"
    if n >= 1e6:
        return f"{n / 1e6:.1f}M"
    if n >= 1e3:
        return f"{n / 1e3:.1f}K"
    return str(n)

def print_losses(losses, model_names, parameters, col_names, cs = 9):
    title = f"MAE Loss\n"
    bound = f"\n{'-'*110}\n\n"
    header1 = f"{' '*20}"+"".join(f"{col_name:<{cs}}" for col_name in col_names)+"\n"
    rows = "".join(
        f"{row_name}: {parameters[i]}\n"
        f"    Training:       {tensor_to_string(losses[i*2], cs)}  >  {losses[i*2].mean():.4f}\n"
        f"    Validation:     {tensor_to_string(losses[i*2+1], cs)}  >  {losses[i*2+1].mean():.4f}\n"
        f"    "
        f"\n"
    for i, row_name in enumerate(model_names))
    output = [
        bound,
        title,
        bound,
        header1,
        rows,
        bound
    ]
    print("".join(output))

In [ ]:
#* REUSES OBJETCS FROM TRAINING
eval_steps = min(eval_bs, len(dls["train"])) + min(eval_bs, len(dls["val"]))
eval_pbar = tqdm(total=3*eval_steps, desc=f"Evaluating the best model parameters...".ljust(80),
                bar_format="|{bar}| {percentage:3.1f}% ({elapsed}) {desc}", position=0, leave=False)
naive_losses = evaluate_model(dls["train"], dls["val"], naiveModel, device, eval_fns, target_weights, eval_bs, eval_pbar)
gpt_losses = evaluate_best_model(stockGPT, device, o1, s1, dls["train"], dls["val"], eval_fns, eval_bs, eval_pbar)
linear_losses = evaluate_best_model(linearModel, device, o2, s2, dls["train"], dls["val"], eval_fns, eval_bs, eval_pbar)

print_losses(process_losses(gpt_losses + linear_losses + naive_losses, "MAE Loss"),
             ["StockGPT", "LinearModel", "NaiveModel"],
             [f"{format_num(stockGPT_params)}", f"{format_num(linearModel_params)}", f"0"],
             StockGPT_cfg["target_features"])

|███▎      | 33.3% (00:13) Evaluating model on validation data... (499/500) [1000/3000]:                  


--------------------------------------------------------------------------------------------------------------

MAE Loss

--------------------------------------------------------------------------------------------------------------

                    vw       ema9     ema20    macd     o        c        h        l        n        rv       
StockGPT: 3.2M
    Training:       0.0043   0.0040   0.0042   0.0139   0.0043   0.0046   0.0049   0.0044   0.0803   0.2687     >  0.0394
    Validation:     0.0042   0.0037   0.0039   0.0119   0.0043   0.0044   0.0046   0.0043   0.0376   0.3020     >  0.0381
    
LinearModel: 6.1K
    Training:       0.0024   0.0007   0.0010   0.0085   0.0025   0.0032   0.0039   0.0040   0.1183   0.2869     >  0.0431
    Validation:     0.0024   0.0006   0.0010   0.0078   0.0025   0.0030   0.0035   0.0039   0.0520   0.3140     >  0.0391
    
NaiveModel: 0
    Training:       0.0017   0.0008   0.0005   0.0108   0.0020   0.0019   0.0019   0.0018   0.1186   0.3729  

|███▎      | 33.3% (00:27) Evaluating model on validation data... (499/500) [1000/3000]: 